# CallPyBack: Advanced Python Callback Framework

This notebook demonstrates the CallPyBack library - an advanced callback decorator framework implementing formal design patterns for Python function observation and monitoring.

## Table of Contents
1. [Setup and Basic Concepts](#setup)
2. [Observer Patterns](#observers)
3. [Error Handling Strategies](#error-handling)
4. [Performance Monitoring](#performance)
5. [Multi-Threading and Concurrency](#threading)
6. [Production Patterns](#production)
7. [Custom Observers](#custom-observers)
8. [Advanced Integration Patterns](#integration)

## 1. Setup and Basic Concepts {#setup}

In [1]:
# Import required modules
import threading
import time
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import random

from callpyback import (
    CallPyBack,
    on_call,
    on_success,
    on_failure,
    on_completion,
    ExecutionContext,
    ExecutionState,
    DefaultErrorHandler,
)
from callpyback.observers.builtin import (
    MetricsObserver,
    LoggingObserver,
    TimingObserver,
)
from callpyback.observers.base import BaseObserver

# Configure logging for cleaner output
logging.basicConfig(
    level=logging.WARNING,
    format="[%(asctime)s] {%(pathname)s:%(lineno)d} %(levelname)s - %(message)s \n\n",
)

### Basic Usage Example

In [2]:
print("=== Basic Callback Patterns ===")

execution_log = []


def track_execution(context):
    execution_log.append(f"{context.function_signature.name}:{context.state.name}")


def log_success(result):
    print(f"✅ Success: {result.value}")


def log_failure(result):
    print(f"❌ Failed: {result.exception}")


@CallPyBack(
    observers=[
        on_call(track_execution),
        on_success(log_success),
        on_failure(log_failure),
    ],
    exception_classes=(ValueError, TypeError),
    default_return="handled_gracefully",
)
def demo_function(operation, value):
    if operation == "success":
        return f"Processed: {value}"
    elif operation == "error":
        raise ValueError(f"Invalid operation: {value}")
    elif operation == "unhandled":
        raise RuntimeError("This won't be caught")
    return "unknown"


# Test cases
test_cases = [("success", "test_data"), ("error", "bad_data"), ("success", "more_data")]

for operation, value in test_cases:
    try:
        result = demo_function(operation, value)
        print(f"Result: {result}")
    except Exception as e:
        print(f"--:[Expection]:-- = Uncaught: {e}")

print(f"Execution log: {execution_log}")

[2025-06-14 18:40:05,680] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: demo_function] [Module: __main__] [Error Type: ValueError] [Error Message: Invalid operation: bad_data] [Arguments: {"operation": "error", "value": "bad_data"}] [Timestamp: 1749919205.68] [Handler Type: Fallback/Catch-all] [Default Return: handled_gracefully] [Stack Trace Available: True] [Action: Returning configured default value: handled_gracefully] 


[2025-06-14 18:40:05,680] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for demo_function | ValueError: Invalid operation: bad_data 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execute_with_observation
    result = func(**arguments)
  File "/tmp/ipykernel_468788/298146486.py", line 27, in demo_f

=== Basic Callback Patterns ===
✅ Success: Processed: test_data
Result: Processed: test_data
❌ Failed: Invalid operation: bad_data
Result: handled_gracefully
✅ Success: Processed: more_data
Result: Processed: more_data
Execution log: ['demo_function:PRE_EXECUTION', 'demo_function:PRE_EXECUTION', 'demo_function:PRE_EXECUTION']


## 2. Observer Patterns {#observers}

### Variable Extraction and State Monitoring

In [3]:
print("\n=== Advanced Observer Patterns ===")

captured_variables = []
state_transitions = []


def capture_variables(local_variables):
    if local_variables:
        captured_variables.append(
            {
                "vars": {
                    k: v
                    for k, v in local_variables.items()
                    if not str(v).startswith("<Variable")
                },
                "timestamp": time.time(),
            }
        )


def track_all_states(context):
    state_transitions.append(
        {
            "state": context.state.name,
            "function": context.function_signature.name,
            "has_result": context.result is not None,
        }
    )


# Observer that monitors all states
from callpyback.observers.callback import CallbackObserver

state_observer = CallbackObserver(
    track_all_states,
    interested_states={
        ExecutionState.PRE_EXECUTION,
        ExecutionState.POST_SUCCESS,
        ExecutionState.POST_FAILURE,
        ExecutionState.COMPLETED,
    },
)


@CallPyBack(
    observers=[state_observer, on_completion(capture_variables)],
    variable_names=["input_data", "processed", "result_type", "output"],
    exception_classes=(ValueError,),
    default_return="processing_failed",
)
def data_pipeline(data, transform="upper"):
    """Complex data processing with variable tracking."""
    input_data = data

    if transform == "upper":
        processed = data.upper() if isinstance(data, str) else str(data).upper()
    elif transform == "reverse":
        processed = data[::-1] if isinstance(data, str) else str(data)[::-1]
    elif transform == "error":
        raise ValueError("Intentional processing error")
    else:
        processed = data

    result_type = type(processed).__name__
    output = f"[{result_type}] {processed}"

    return output


# Test the pipeline
test_data = [
    ("hello world", "upper"),
    ("python", "reverse"),
    (12345, "upper"),
    ("test", "error"),
    ("final", "unknown"),
]

for data, transform in test_data:
    result = data_pipeline(data, transform)
    print(f"Input: {data}, Transform: {transform} -> {result}")

print(f"\nCaptured {len(captured_variables)} variable snapshots")
print(f"State transitions: {len(state_transitions)}")

[2025-06-14 18:40:05,751] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: data_pipeline] [Module: __main__] [Error Type: ValueError] [Error Message: Intentional processing error] [Arguments: {"data": "test", "transform": "error"}] [Timestamp: 1749919205.75] [Handler Type: Fallback/Catch-all] [Default Return: processing_failed] [Stack Trace Available: True] [Action: Returning configured default value: processing_failed] 


[2025-06-14 18:40:05,752] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for data_pipeline | ValueError: Intentional processing error 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execute_with_observation
    result = func(**arguments)
  File "/tmp/ipykernel_468788/3917102099.py", line 48, in data_pipel


=== Advanced Observer Patterns ===
Input: hello world, Transform: upper -> [str] HELLO WORLD
Input: python, Transform: reverse -> [str] nohtyp
Input: 12345, Transform: upper -> [str] 12345
Input: test, Transform: error -> processing_failed
Input: final, Transform: unknown -> [str] final

Captured 5 variable snapshots
State transitions: 15


## 3. Error Handling Strategies {#error-handling}

### Production-Ready Error Chains

In [4]:
print("\n=== Error Handling Strategies ===")

# Custom error tracking
error_stats = defaultdict(int)


def track_errors(result):
    error_type = type(result.exception).__name__
    error_stats[error_type] += 1
    print(f"🚨 Tracked {error_type}: {result.exception}")


# Strategy 1: Graceful degradation
@CallPyBack(
    observers=[on_failure(track_errors)],
    exception_classes=(ValueError, TypeError, ConnectionError),
    default_return={"status": "error", "fallback": True},
)
def resilient_service(operation, data):
    """Service with graceful error handling."""
    if operation == "validate":
        if not isinstance(data, str):
            raise TypeError(f"Expected string, got {type(data)}")
        if len(data) < 3:
            raise ValueError("Data too short")
    elif operation == "network":
        raise ConnectionError("Service temporarily unavailable")
    elif operation == "critical":
        raise RuntimeError("Critical system error")  # Won't be caught

    return {"status": "success", "data": f"Processed {data}"}


# Strategy 2: Circuit breaker pattern
failure_count = {"value": 0}
circuit_open = {"state": False}


def circuit_breaker(context):
    if circuit_open["state"]:
        print("🔴 Circuit breaker OPEN - blocking execution")
        return "circuit_open"


def track_failures(result):
    failure_count["value"] += 1
    if failure_count["value"] >= 3:
        circuit_open["state"] = True
        print(f"🔴 Circuit breaker OPENED after {failure_count['value']} failures")


@CallPyBack(
    observers=[on_call(circuit_breaker), on_failure(track_failures)],
    exception_classes=(Exception,),
    default_return="circuit_breaker_active",
)
def circuit_protected_service(should_fail=False):
    if circuit_open["state"]:
        return "service_unavailable"

    if should_fail:
        raise ConnectionError("Service error")
    return "service_response"


# Test error handling strategies
print("Testing resilient service:")
test_cases = [
    ("validate", "hello"),  # Success
    ("validate", 123),  # Type error
    ("network", "data"),  # Network error
    ("critical", "data"),  # Critical error (uncaught)
]

for operation, data in test_cases:
    try:
        result = resilient_service(operation, data)
        print(f"  {operation}: {result}")
    except Exception as e:
        print(f"  {operation}: UNCAUGHT - {e}")

print(f"\nError statistics: {dict(error_stats)}")

print("\nTesting circuit breaker:")
for i in range(5):
    result = circuit_protected_service(should_fail=True)
    print(f"  Attempt {i+1}: {result}")

[2025-06-14 18:40:05,810] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: resilient_service] [Module: __main__] [Error Type: TypeError] [Error Message: Expected string, got <class 'int'>] [Arguments: {"operation": "validate", "data": 123}] [Timestamp: 1749919205.81] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'error', 'fallback': True}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'error', 'fallback': True}] 


[2025-06-14 18:40:05,810] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for resilient_service | TypeError: Expected string, got <class 'int'> 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execute_with_observation
    result = func(**arguments)
  File "/t


=== Error Handling Strategies ===
Testing resilient service:
  validate: {'status': 'success', 'data': 'Processed hello'}
🚨 Tracked TypeError: Expected string, got <class 'int'>
  validate: {'status': 'error', 'fallback': True}
🚨 Tracked ConnectionError: Service temporarily unavailable
  network: {'status': 'error', 'fallback': True}
  critical: UNCAUGHT - Critical system error

Error statistics: {'TypeError': 1, 'ConnectionError': 1}

Testing circuit breaker:
  Attempt 1: circuit_breaker_active
  Attempt 2: circuit_breaker_active
🔴 Circuit breaker OPENED after 3 failures
  Attempt 3: circuit_breaker_active
🔴 Circuit breaker OPEN - blocking execution
  Attempt 4: service_unavailable
🔴 Circuit breaker OPEN - blocking execution
  Attempt 5: service_unavailable


## 4. Performance Monitoring {#performance}

### Advanced Performance Tracking

In [5]:
print("\n=== Performance Monitoring ===")


# Custom performance profiler
class AdvancedPerformanceProfiler(BaseObserver):
    def __init__(self, percentile_threshold=95):
        super().__init__(priority=100, name="AdvancedProfiler")
        self.execution_times = defaultdict(list)
        self.percentile_threshold = percentile_threshold
        self.slow_executions = []

    def update(self, context):
        if (
            context.state == ExecutionState.POST_SUCCESS
            and context.result
            and hasattr(context.result, "execution_time")
        ):

            func_name = context.function_signature.name
            exec_time = context.result.execution_time

            self.execution_times[func_name].append(exec_time)

            # Check for performance anomalies
            if len(self.execution_times[func_name]) >= 5:
                times = self.execution_times[func_name]
                threshold = self._calculate_percentile(times, self.percentile_threshold)

                if exec_time > threshold:
                    alert = {
                        "function": func_name,
                        "time": exec_time,
                        "threshold": threshold,
                        "args": context.arguments,
                    }
                    self.slow_executions.append(alert)
                    print(
                        f"🐌 PERFORMANCE ALERT: {func_name} took {exec_time*1000:.1f}ms "
                        f"(>{self.percentile_threshold}th percentile: {threshold*1000:.1f}ms)"
                    )

    def _calculate_percentile(self, values, percentile):
        sorted_values = sorted(values)
        index = int((percentile / 100) * len(sorted_values))
        return sorted_values[min(index, len(sorted_values) - 1)]

    def get_stats(self):
        return {
            "total_functions": len(self.execution_times),
            "total_executions": sum(
                len(times) for times in self.execution_times.values()
            ),
            "slow_executions": len(self.slow_executions),
            "avg_times": {
                func: sum(times) / len(times)
                for func, times in self.execution_times.items()
            },
        }


# Setup monitoring
profiler = AdvancedPerformanceProfiler(percentile_threshold=90)
timing_observer = TimingObserver(threshold=0.05)  # 50ms threshold


@CallPyBack(observers=[profiler, timing_observer])
def variable_performance_task(complexity_level):
    """Task with variable performance characteristics."""
    # Simulate different performance profiles
    if complexity_level == "fast":
        time.sleep(0.01)  # 10ms
    elif complexity_level == "medium":
        time.sleep(0.03)  # 30ms
    elif complexity_level == "slow":
        time.sleep(0.08)  # 80ms
    elif complexity_level == "variable":
        time.sleep(random.uniform(0.01, 0.1))  # 10-100ms

    return f"Completed {complexity_level} task"


# Run performance tests
test_profile = ["fast"] * 10 + ["medium"] * 8 + ["slow"] * 5 + ["variable"] * 12

print("Running performance tests...")
for i, complexity in enumerate(test_profile):
    result = variable_performance_task(complexity)
    if i % 5 == 0:
        print(f"  Completed {i+1}/{len(test_profile)} tests")

# Show performance statistics
stats = profiler.get_stats()
print(f"\nPerformance Statistics:")
print(f"  Total executions: {stats['total_executions']}")
print(f"  Slow executions detected: {stats['slow_executions']}")
print(f"  Average execution times:")
for func, avg_time in stats["avg_times"].items():
    print(f"    {func}: {avg_time*1000:.1f}ms")

slow_executions = timing_observer.get_slow_executions()
print(f"  Timing observer detected {len(slow_executions)} slow executions")


=== Performance Monitoring ===
Running performance tests...
  Completed 1/35 tests
  Completed 6/35 tests
  Completed 11/35 tests
  Completed 16/35 tests


[2025-06-14 18:40:06,300] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.080s (threshold: 0.05s) 


[2025-06-14 18:40:06,381] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.080s (threshold: 0.05s) 


[2025-06-14 18:40:06,463] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.080s (threshold: 0.05s) 


[2025-06-14 18:40:06,544] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.080s (threshold: 0.05s) 


[2025-06-14 18:40:06,624] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.080s (threshold: 0.05s) 




  Completed 21/35 tests


[2025-06-14 18:40:06,707] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.081s (threshold: 0.05s) 


[2025-06-14 18:40:06,813] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.089s (threshold: 0.05s) 


[2025-06-14 18:40:06,873] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.059s (threshold: 0.05s) 


[2025-06-14 18:40:07,011] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.094s (threshold: 0.05s) 




  Completed 26/35 tests


[2025-06-14 18:40:07,069] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.057s (threshold: 0.05s) 


[2025-06-14 18:40:07,202] {/home/adhd/src/personal/observer-pattern/callpyback/observers/builtin.py:133} WARNING - Slow execution detected: variable_performance_task took 0.073s (threshold: 0.05s) 




  Completed 31/35 tests

Performance Statistics:
  Total executions: 0
  Slow executions detected: 0
  Average execution times:
  Timing observer detected 11 slow executions


## 5. Multi-Threading and Concurrency {#threading}

### Thread-Safe Callback Execution

In [6]:
# Thread-safe metrics collection
thread_metrics = {
    "executions": defaultdict(int),
    "errors": defaultdict(int),
    "total_time": defaultdict(float),
    "lock": threading.RLock(),
}


def thread_safe_metrics(context):
    with thread_metrics["lock"]:
        thread_id = threading.current_thread().name
        thread_metrics["executions"][thread_id] += 1


def track_thread_success(result):
    with thread_metrics["lock"]:
        thread_id = threading.current_thread().name
        thread_metrics["total_time"][thread_id] += result.execution_time


def track_thread_failure(result):
    with thread_metrics["lock"]:
        thread_id = threading.current_thread().name
        thread_metrics["errors"][thread_id] += 1


# Pattern 1: Concurrent data processing
@CallPyBack(
    observers=[
        on_call(thread_safe_metrics),
        on_success(track_thread_success),
        on_failure(track_thread_failure),
    ],
    exception_classes=(ValueError, ConnectionError),
    default_return={"status": "error", "thread_safe": True},
)
def concurrent_data_processor(data_chunk, processing_mode="normal"):
    """Thread-safe data processing function."""
    thread_id = threading.current_thread().name

    # Simulate variable processing time
    processing_time = random.uniform(0.01, 0.1)
    time.sleep(processing_time)

    if processing_mode == "error" and random.random() < 0.2:
        raise ValueError(f"Processing error in {thread_id}")

    if processing_mode == "network_error" and random.random() < 0.15:
        raise ConnectionError(f"Network error in {thread_id}")

    # Simulate data processing
    return {
        "thread": thread_id,
        "processed_data": f"processed_{data_chunk}",
        "processing_time": processing_time,
        "timestamp": time.time(),
    }


# Execute concurrent tests
print("Testing concurrent data processing...")

# Test 1: ThreadPoolExecutor with mixed workloads
data_chunks = [f"chunk_{i}" for i in range(20)]
processing_modes = ["normal"] * 12 + ["error"] * 4 + ["network_error"] * 4

with ThreadPoolExecutor(max_workers=6, thread_name_prefix="DataWorker") as executor:
    # Submit data processing tasks
    futures = []
    for chunk, mode in zip(data_chunks, processing_modes):
        future = executor.submit(concurrent_data_processor, chunk, mode)
        futures.append(future)

    # Collect results with timeout
    successful_results = []
    for future in as_completed(futures, timeout=10):
        try:
            result = future.result()
            successful_results.append(result)
        except Exception as e:
            print(f"Task failed: {e}")

print(f"Data processing completed: {len(successful_results)} successful")

Testing concurrent data processing...
Data processing completed: 20 successful


In [7]:
# Pattern 2: Producer-Consumer with callbacks
work_queue = []
results_queue = []
queue_lock = threading.Lock()


def track_work_completion(result):
    with queue_lock:
        if hasattr(result, "value") and isinstance(result.value, dict):
            results_queue.append(result.value)


@CallPyBack(
    observers=[on_success(track_work_completion)],
    exception_classes=(Exception,),
    default_return={"status": "failed", "retry": True},
)
def worker_task(task_id, work_type="compute"):
    """Worker task with callback tracking."""

    if work_type == "compute":
        # CPU-bound work simulation
        time.sleep(random.uniform(0.02, 0.08))
        return {"task_id": task_id, "result": f"computed_{task_id}", "type": work_type}

    elif work_type == "io":
        # I/O-bound work simulation
        time.sleep(random.uniform(0.05, 0.15))
        return {"task_id": task_id, "result": f"io_result_{task_id}", "type": work_type}

    elif work_type == "error":
        if random.random() < 0.3:
            raise RuntimeError(f"Task {task_id} failed")
        return {"task_id": task_id, "result": "error_handled", "type": work_type}


# Test 2: Producer-Consumer pattern
print("Testing producer-consumer pattern...")


def producer_worker():
    """Producer thread function."""
    tasks = [(i, random.choice(["compute", "io", "error"])) for i in range(15)]

    with ThreadPoolExecutor(max_workers=4, thread_name_prefix="Consumer") as executor:
        futures = [
            executor.submit(worker_task, task_id, work_type)
            for task_id, work_type in tasks
        ]

        for future in as_completed(futures):
            try:
                result = future.result()
            except Exception:
                pass  # Errors are handled by callbacks


# Run producer
producer_thread = threading.Thread(target=producer_worker, name="Producer")
producer_thread.start()
producer_thread.join()

[2025-06-14 18:40:07,487] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: worker_task] [Module: __main__] [Error Type: RuntimeError] [Error Message: Task 4 failed] [Arguments: {"task_id": 4, "work_type": "error"}] [Timestamp: 1749919207.49] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'failed', 'retry': True}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'failed', 'retry': True}] 


[2025-06-14 18:40:07,489] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for worker_task | RuntimeError: Task 4 failed 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execute_with_observation
    result = func(**arguments)
  File "/tmp/ipykernel_468788/3485250068.py", line 31, in worker

Testing producer-consumer pattern...


In [8]:
# Display thread metrics
with thread_metrics["lock"]:
    print(f"\nThread Execution Metrics:")
    print(f"  Threads that executed tasks: {len(thread_metrics['executions'])}")

    for thread_name in sorted(thread_metrics["executions"].keys()):
        executions = thread_metrics["executions"][thread_name]
        errors = thread_metrics["errors"][thread_name]
        total_time = thread_metrics["total_time"][thread_name]
        avg_time = total_time / max(executions, 1)

        print(f"  {thread_name}:")
        print(f"    Executions: {executions}")
        print(f"    Errors: {errors}")
        print(f"    Avg time: {avg_time*1000:.1f}ms")
        print(f"    Success rate: {((executions-errors)/max(executions,1))*100:.1f}%")

print(f"  Producer-Consumer results: {len(results_queue)} completed tasks")


Thread Execution Metrics:
  Threads that executed tasks: 6
  DataWorker_0:
    Executions: 3
    Errors: 0
    Avg time: 56.6ms
    Success rate: 100.0%
  DataWorker_1:
    Executions: 4
    Errors: 0
    Avg time: 42.7ms
    Success rate: 100.0%
  DataWorker_2:
    Executions: 3
    Errors: 0
    Avg time: 65.5ms
    Success rate: 100.0%
  DataWorker_3:
    Executions: 4
    Errors: 0
    Avg time: 55.9ms
    Success rate: 100.0%
  DataWorker_4:
    Executions: 3
    Errors: 0
    Avg time: 56.9ms
    Success rate: 100.0%
  DataWorker_5:
    Executions: 3
    Errors: 0
    Avg time: 67.3ms
    Success rate: 100.0%
  Producer-Consumer results: 12 completed tasks


### Race Condition Detection and Mitigation

In [9]:
print("\n=== Race Condition Testing ===")

# Shared resource with potential race conditions
shared_counter = {"value": 0, "operations": []}
race_detector = {"conflicts": 0, "lock": threading.Lock()}


def detect_race_conditions(context):
    """Observer to detect potential race conditions."""
    thread_id = threading.current_thread().name
    timestamp = time.time()

    # Check for rapid successive access
    with race_detector["lock"]:
        recent_ops = [
            op
            for op in shared_counter["operations"]
            if timestamp - op["timestamp"] < 0.001
        ]  # 1ms window

        if len(recent_ops) > 1:
            race_detector["conflicts"] += 1
            print(
                f"⚠️  Potential race condition detected! {len(recent_ops)} concurrent operations"
            )


def track_operations(context):
    """Track all operations on shared resource."""
    thread_id = threading.current_thread().name
    operation = {
        "thread": thread_id,
        "timestamp": time.time(),
        "function": context.function_signature.name,
    }
    shared_counter["operations"].append(operation)


# Unsafe operation (no locking)
@CallPyBack(
    observers=[on_call(detect_race_conditions), on_call(track_operations)],
    exception_classes=(Exception,),
    default_return="operation_failed",
)
def unsafe_increment(amount=1):
    """Unsafe increment operation - prone to race conditions."""
    current = shared_counter["value"]
    # Simulate processing time that can cause race conditions
    time.sleep(0.001)
    shared_counter["value"] = current + amount
    return shared_counter["value"]


# Safe operation (with locking)
increment_lock = threading.Lock()


@CallPyBack(
    observers=[on_call(detect_race_conditions), on_call(track_operations)],
    exception_classes=(Exception,),
    default_return="safe_operation_failed",
)
def safe_increment(amount=1):
    """Thread-safe increment operation."""
    with increment_lock:
        current = shared_counter["value"]
        time.sleep(0.001)  # Same processing time
        shared_counter["value"] = current + amount
        return shared_counter["value"]


# Test race conditions
print("Testing unsafe operations (race conditions expected)...")
shared_counter["value"] = 0

with ThreadPoolExecutor(max_workers=10, thread_name_prefix="RaceTest") as executor:
    # Submit many concurrent unsafe operations
    futures = [executor.submit(unsafe_increment, 1) for _ in range(50)]
    results = [f.result() for f in futures]

unsafe_final = shared_counter["value"]
unsafe_conflicts = race_detector["conflicts"]

print(f"Unsafe operations completed:")
print(f"  Expected value: 50, Actual value: {unsafe_final}")
print(f"  Race conditions detected: {unsafe_conflicts}")
print(f"  Data integrity: {'✅ OK' if unsafe_final == 50 else '❌ CORRUPTED'}")

# Reset for safe test
shared_counter["value"] = 0
race_detector["conflicts"] = 0

print("\nTesting safe operations (no race conditions expected)...")

with ThreadPoolExecutor(max_workers=10, thread_name_prefix="SafeTest") as executor:
    # Submit many concurrent safe operations
    futures = [executor.submit(safe_increment, 1) for _ in range(50)]
    results = [f.result() for f in futures]

safe_final = shared_counter["value"]
safe_conflicts = race_detector["conflicts"]

print(f"Safe operations completed:")
print(f"  Expected value: 50, Actual value: {safe_final}")
print(f"  Race conditions detected: {safe_conflicts}")
print(f"  Data integrity: {'✅ OK' if safe_final == 50 else '❌ CORRUPTED'}")

#    return {
#        "unsafe_result": unsafe_final,
#        "safe_result": safe_final,
#        "conflicts_detected": unsafe_conflicts + safe_conflicts
#    }


=== Race Condition Testing ===
Testing unsafe operations (race conditions expected)...
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 3 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition detected! 3 concurrent operations
⚠️  Potential race condition detected! 2 concurrent operations
⚠️  Potential race condition d

## 6. Production Patterns {#production}

### Enterprise-Grade Monitoring and Auditing

In [10]:
print("\n=== Production Patterns ===")


# Enterprise audit logger
class EnterpriseAuditObserver(BaseObserver):
    def __init__(self):
        super().__init__(priority=100, name="EnterpriseAudit")
        self.audit_log = []
        self.security_events = []
        self.performance_alerts = []

    def update(self, context):
        if context.state == ExecutionState.COMPLETED:
            # Standard audit entry
            audit_entry = {
                "timestamp": context.timestamp,
                "function": context.function_signature.name,
                "user": context.arguments.get("user_id", "system"),
                "operation": context.arguments.get("operation", "unknown"),
                "success": context.is_successful,
                "execution_time": (
                    getattr(context.result, "execution_time", 0)
                    if context.result
                    else 0
                ),
            }

            # Add error details if failed
            if context.is_failed:
                audit_entry["error"] = str(context.result.exception)
                audit_entry["error_type"] = context.result.exception_type.__name__

            self.audit_log.append(audit_entry)

            # Security event detection
            if self._is_security_event(context):
                security_event = {
                    **audit_entry,
                    "security_level": "HIGH",
                    "requires_review": True,
                }
                self.security_events.append(security_event)
                print(
                    f"🔒 SECURITY EVENT: {context.function_signature.name} by {audit_entry['user']}"
                )

            # Performance alert detection
            exec_time = audit_entry["execution_time"]
            if exec_time > 0.1:  # 100ms threshold
                perf_alert = {
                    **audit_entry,
                    "alert_type": "SLOW_EXECUTION",
                    "threshold_exceeded": exec_time,
                }
                self.performance_alerts.append(perf_alert)
                print(
                    f"🐌 PERFORMANCE ALERT: {context.function_signature.name} took {exec_time*1000:.0f}ms"
                )

    def _is_security_event(self, context):
        """Detect security-relevant events."""
        sensitive_operations = ["delete", "admin", "security", "auth"]
        operation = context.arguments.get("operation", "").lower()
        return any(sens_op in operation for sens_op in sensitive_operations)

    def get_audit_summary(self):
        total_operations = len(self.audit_log)
        successful_ops = sum(1 for entry in self.audit_log if entry["success"])
        failed_ops = total_operations - successful_ops

        return {
            "total_operations": total_operations,
            "successful_operations": successful_ops,
            "failed_operations": failed_ops,
            "security_events": len(self.security_events),
            "performance_alerts": len(self.performance_alerts),
            "success_rate": (successful_ops / max(total_operations, 1)) * 100,
        }


# Setup enterprise monitoring
audit_observer = EnterpriseAuditObserver()
metrics_observer = MetricsObserver()

# Circuit breaker implementation
circuit_breaker_state = {
    "failures": 0,
    "last_failure": 0,
    "state": "CLOSED",  # CLOSED, OPEN, HALF_OPEN
    "threshold": 5,
    "timeout": 10,  # seconds
}


def circuit_breaker_check(context):
    """Implement circuit breaker pattern."""
    current_time = time.time()

    if circuit_breaker_state["state"] == "OPEN":
        if (
            current_time - circuit_breaker_state["last_failure"]
            > circuit_breaker_state["timeout"]
        ):
            circuit_breaker_state["state"] = "HALF_OPEN"
            print("🔄 Circuit breaker entering HALF_OPEN state")
        else:
            print("🔴 Circuit breaker OPEN - request blocked")
            return "circuit_breaker_open"


def circuit_breaker_success(result):
    """Handle successful execution."""
    if circuit_breaker_state["state"] == "HALF_OPEN":
        circuit_breaker_state["state"] = "CLOSED"
        circuit_breaker_state["failures"] = 0
        print("✅ Circuit breaker CLOSED - service recovered")


def circuit_breaker_failure(result):
    """Handle failed execution."""
    circuit_breaker_state["failures"] += 1
    circuit_breaker_state["last_failure"] = time.time()

    if circuit_breaker_state["failures"] >= circuit_breaker_state["threshold"]:
        circuit_breaker_state["state"] = "OPEN"
        print(f"🔴 Circuit breaker OPEN - {circuit_breaker_state['failures']} failures")


# Production service with full monitoring
@CallPyBack(
    observers=[
        audit_observer,
        metrics_observer,
        on_call(circuit_breaker_check),
        on_success(circuit_breaker_success),
        on_failure(circuit_breaker_failure),
    ],
    exception_classes=(ValueError, ConnectionError, TimeoutError, PermissionError),
    default_return={"status": "error", "service": "unavailable"},
)
def enterprise_service(operation, user_id, data, **kwargs):
    """Enterprise service with comprehensive monitoring."""

    # Simulate different operation types
    if operation == "read":
        time.sleep(random.uniform(0.01, 0.05))
        return {"status": "success", "data": f"read_result_{data.get('id', 'unknown')}"}

    elif operation == "write":
        time.sleep(random.uniform(0.02, 0.08))
        return {"status": "success", "written": data}

    elif operation == "delete":
        if user_id != "admin_user":
            raise PermissionError("Insufficient permissions for delete operation")
        time.sleep(random.uniform(0.01, 0.03))
        return {"status": "success", "deleted": data.get("id")}

    elif operation == "admin_operation":
        if not user_id.startswith("admin"):
            raise PermissionError("Admin privileges required")
        time.sleep(random.uniform(0.05, 0.15))  # Slow admin operation
        return {"status": "success", "admin_result": "completed"}

    elif operation == "network_dependent":
        if random.random() < 0.3:  # 30% failure rate
            raise ConnectionError("External service unavailable")
        time.sleep(random.uniform(0.03, 0.12))
        return {"status": "success", "external_data": "fetched"}

    elif operation == "validation_check":
        if not data.get("valid", True):
            raise ValueError("Data validation failed")
        return {"status": "success", "validated": True}

    return {"status": "success", "operation": operation}


# Production test scenarios
test_scenarios = [
    # Normal operations
    ("read", "user123", {"id": "doc1"}),
    ("write", "user456", {"content": "test data"}),
    ("read", "user789", {"id": "doc2"}),
    # Security-sensitive operations
    ("delete", "admin_user", {"id": "doc1"}),
    ("delete", "regular_user", {"id": "doc2"}),  # Should fail
    ("admin_operation", "admin_user", {}),
    ("admin_operation", "regular_user", {}),  # Should fail
    # Network-dependent operations (some will fail)
    *[("network_dependent", f"user{i}", {}) for i in range(5)],
    # Validation operations
    ("validation_check", "user123", {"valid": True}),
    ("validation_check", "user456", {"valid": False}),  # Should fail
    # More network operations to potentially trigger circuit breaker
    *[("network_dependent", f"user{i}", {}) for i in range(8)],
]

print("Executing production test scenarios...")
results = []

for i, (operation, user_id, data) in enumerate(test_scenarios):
    result = enterprise_service(operation, user_id, data)
    results.append(result)

    if i % 5 == 0:
        print(f"  Completed {i+1}/{len(test_scenarios)} operations")

# Generate comprehensive report
audit_summary = audit_observer.get_audit_summary()
metrics_data = metrics_observer.get_metrics()

print(f"\n=== Production Service Report ===")
print(f"Total Operations: {audit_summary['total_operations']}")
print(f"Success Rate: {audit_summary['success_rate']:.1f}%")
print(f"Security Events: {audit_summary['security_events']}")
print(f"Performance Alerts: {audit_summary['performance_alerts']}")
print(f"Circuit Breaker State: {circuit_breaker_state['state']}")
print(f"Average Execution Time: {metrics_data['average_execution_time']*1000:.1f}ms")


=== Production Patterns ===
Executing production test scenarios...
  Completed 1/22 operations


[2025-06-14 18:40:08,073] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: enterprise_service] [Module: __main__] [Error Type: PermissionError] [Error Message: Insufficient permissions for delete operation] [Arguments: {"operation": "delete", "user_id": "regular_user", "data": {"id": "doc2"}, "kwargs": {}}] [Timestamp: 1749919208.07] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'error', 'service': 'unavailable'}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'error', 'service': 'unavailable'}] 


[2025-06-14 18:40:08,074] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for enterprise_service | PermissionError: Insufficient permissions for delete operation 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpybac

🔒 SECURITY EVENT: enterprise_service by admin_user
🔒 SECURITY EVENT: enterprise_service by regular_user


[2025-06-14 18:40:08,149] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: enterprise_service] [Module: __main__] [Error Type: PermissionError] [Error Message: Admin privileges required] [Arguments: {"operation": "admin_operation", "user_id": "regular_user", "data": {}, "kwargs": {}}] [Timestamp: 1749919208.15] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'error', 'service': 'unavailable'}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'error', 'service': 'unavailable'}] 


[2025-06-14 18:40:08,150] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for enterprise_service | PermissionError: Admin privileges required 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execute

🔒 SECURITY EVENT: enterprise_service by admin_user
  Completed 6/22 operations
🔒 SECURITY EVENT: enterprise_service by regular_user


...

  Completed 11/22 operations


[2025-06-14 18:40:08,374] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: enterprise_service] [Module: __main__] [Error Type: ValueError] [Error Message: Data validation failed] [Arguments: {"operation": "validation_check", "user_id": "user456", "data": {"valid": false}, "kwargs": {}}] [Timestamp: 1749919208.37] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'error', 'service': 'unavailable'}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'error', 'service': 'unavailable'}] 


[2025-06-14 18:40:08,375] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for enterprise_service | ValueError: Data validation failed 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execute_with_

🔴 Circuit breaker OPEN - 5 failures
🔴 Circuit breaker OPEN - request blocked
🔴 Circuit breaker OPEN - request blocked
  Completed 16/22 operations
🔴 Circuit breaker OPEN - request blocked
🔴 Circuit breaker OPEN - 6 failures
🔴 Circuit breaker OPEN - request blocked


[2025-06-14 18:40:08,564] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: enterprise_service] [Module: __main__] [Error Type: ConnectionError] [Error Message: External service unavailable] [Arguments: {"operation": "network_dependent", "user_id": "user4", "data": {}, "kwargs": {}}] [Timestamp: 1749919208.56] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'error', 'service': 'unavailable'}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'error', 'service': 'unavailable'}] 


[2025-06-14 18:40:08,565] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for enterprise_service | ConnectionError: External service unavailable 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in _execut

🔴 Circuit breaker OPEN - request blocked
🔴 Circuit breaker OPEN - 7 failures
🔴 Circuit breaker OPEN - request blocked
🔴 Circuit breaker OPEN - 8 failures
🔴 Circuit breaker OPEN - request blocked
  Completed 21/22 operations
🔴 Circuit breaker OPEN - request blocked
🐌 PERFORMANCE ALERT: enterprise_service took 101ms

=== Production Service Report ===
Total Operations: 22
Success Rate: 63.6%
Security Events: 4
Performance Alerts: 1
Circuit Breaker State: OPEN
Average Execution Time: 59.3ms


## 7. Custom Observers {#custom-observers}

### Building Domain-Specific Observers

In [11]:
print("\n=== Custom Domain-Specific Observers ===")


# Custom observer for machine learning workflow
class MLWorkflowObserver(BaseObserver):
    def __init__(self):
        super().__init__(priority=90, name="MLWorkflow")
        self.model_metrics = {}
        self.training_history = []
        self.inference_stats = defaultdict(list)

    def update(self, context):
        if context.state == ExecutionState.COMPLETED:
            operation = context.arguments.get("operation", "unknown")

            if operation == "train":
                self._handle_training(context)
            elif operation == "predict":
                self._handle_inference(context)
            elif operation == "evaluate":
                self._handle_evaluation(context)

    def _handle_training(self, context):
        """Handle model training events."""
        if context.is_successful and context.local_variables:
            model_metrics = {
                "accuracy": context.local_variables.get("accuracy", 0),
                "loss": context.local_variables.get("loss", float("inf")),
                "epoch": context.local_variables.get("epoch", 0),
                "timestamp": context.timestamp,
            }
            self.training_history.append(model_metrics)
            print(
                f"📊 Training: Epoch {model_metrics['epoch']}, "
                f"Accuracy: {model_metrics['accuracy']:.3f}, "
                f"Loss: {model_metrics['loss']:.3f}"
            )

    def _handle_inference(self, context):
        """Handle model inference events."""
        if context.result and hasattr(context.result, "execution_time"):
            model_name = context.arguments.get("model_name", "default")
            inference_time = context.result.execution_time
            self.inference_stats[model_name].append(inference_time)

            # Alert on slow inference
            if inference_time > 0.1:
                print(
                    f"🔍 SLOW INFERENCE: {model_name} took {inference_time*1000:.0f}ms"
                )

    def _handle_evaluation(self, context):
        """Handle model evaluation events."""
        if context.is_successful and context.result:
            metrics = context.result.value if hasattr(context.result, "value") else {}
            self.model_metrics.update(metrics)
            print(f"📈 Evaluation: {metrics}")

    def get_ml_summary(self):
        avg_inference_times = {
            model: sum(times) / len(times)
            for model, times in self.inference_stats.items()
            if times
        }

        return {
            "training_epochs": len(self.training_history),
            "models_evaluated": len(self.model_metrics),
            "inference_models": len(self.inference_stats),
            "avg_inference_times": avg_inference_times,
            "latest_metrics": self.model_metrics,
        }


# Custom observer for financial transactions
class FinancialAuditObserver(BaseObserver):
    def __init__(self):
        super().__init__(priority=95, name="FinancialAudit")
        self.transactions = []
        self.suspicious_activity = []
        self.daily_totals = defaultdict(float)

    def update(self, context):
        if context.state == ExecutionState.COMPLETED and context.arguments.get(
            "transaction_type"
        ):

            self._audit_transaction(context)

    def _audit_transaction(self, context):
        """Audit financial transaction."""
        transaction = {
            "timestamp": context.timestamp,
            "type": context.arguments.get("transaction_type"),
            "amount": context.arguments.get("amount", 0),
            "user_id": context.arguments.get("user_id"),
            "success": context.is_successful,
            "reference": f"TXN_{len(self.transactions)+1:06d}",
        }

        if context.is_failed:
            transaction["failure_reason"] = str(context.result.exception)

        self.transactions.append(transaction)

        # Track daily totals
        day_key = time.strftime("%Y-%m-%d", time.localtime(context.timestamp))
        if transaction["success"]:
            self.daily_totals[day_key] += transaction["amount"]

        # Detect suspicious activity
        if self._is_suspicious(transaction):
            self.suspicious_activity.append(transaction)
            print(
                f"🚨 SUSPICIOUS TRANSACTION: {transaction['reference']} "
                f"- ${transaction['amount']} by {transaction['user_id']}"
            )

    def _is_suspicious(self, transaction):
        """Detect suspicious transaction patterns."""
        # Large transaction amount
        if transaction["amount"] > 10000:
            return True

        # High frequency from same user
        user_recent = [
            t for t in self.transactions[-10:] if t["user_id"] == transaction["user_id"]
        ]
        if len(user_recent) >= 5:
            return True

        return False

    def get_financial_summary(self):
        total_transactions = len(self.transactions)
        successful_transactions = sum(1 for t in self.transactions if t["success"])
        total_amount = sum(t["amount"] for t in self.transactions if t["success"])

        return {
            "total_transactions": total_transactions,
            "successful_transactions": successful_transactions,
            "total_amount": total_amount,
            "suspicious_activity": len(self.suspicious_activity),
            "daily_totals": dict(self.daily_totals),
            "success_rate": (successful_transactions / max(total_transactions, 1))
            * 100,
        }


# Setup custom observers
ml_observer = MLWorkflowObserver()
financial_observer = FinancialAuditObserver()


# ML workflow functions
@CallPyBack(
    observers=[ml_observer],
    variable_names=["accuracy", "loss", "epoch"],
    exception_classes=(ValueError, RuntimeError),
    default_return={"status": "training_failed"},
)
def ml_training_step(operation, model_name, epoch_data):
    """ML training step with monitoring."""
    epoch = epoch_data.get("epoch", 0)

    if operation == "train":
        # Simulate training metrics
        accuracy = min(0.95, 0.6 + (epoch * 0.05) + random.uniform(-0.1, 0.1))
        loss = max(0.05, 2.0 - (epoch * 0.15) + random.uniform(-0.2, 0.2))

        if accuracy < 0.5:
            raise ValueError("Training diverged - accuracy too low")

        return {"accuracy": accuracy, "loss": loss, "epoch": epoch}

    elif operation == "predict":
        # Simulate inference
        prediction_time = random.uniform(0.01, 0.2)
        time.sleep(prediction_time)
        return {
            "prediction": f"class_{random.randint(0, 9)}",
            "confidence": random.uniform(0.7, 0.99),
        }

    elif operation == "evaluate":
        metrics = {
            "accuracy": random.uniform(0.85, 0.95),
            "precision": random.uniform(0.80, 0.90),
            "recall": random.uniform(0.75, 0.90),
            "f1_score": random.uniform(0.78, 0.92),
        }
        return metrics


# Financial transaction function
@CallPyBack(
    observers=[financial_observer],
    exception_classes=(ValueError, PermissionError, RuntimeError),
    default_return={"status": "transaction_failed", "amount": 0},
)
def financial_transaction(transaction_type, user_id, amount, **kwargs):
    """Financial transaction with audit trail."""

    if amount <= 0:
        raise ValueError("Transaction amount must be positive")

    if transaction_type == "withdrawal" and amount > 50000:
        raise PermissionError("Withdrawal limit exceeded")

    # Simulate transaction processing
    processing_time = random.uniform(0.01, 0.05)
    time.sleep(processing_time)

    # Simulate occasional failures
    if random.random() < 0.05:  # 5% failure rate
        raise RuntimeError("Transaction processing error")

    return {
        "status": "success",
        "transaction_type": transaction_type,
        "amount": amount,
        "processed_at": time.time(),
    }


# Run ML workflow simulation
print("Running ML workflow simulation...")

# Training simulation
for epoch in range(10):
    result = ml_training_step("train", "sentiment_classifier", {"epoch": epoch})
    if isinstance(result, dict) and "accuracy" in result:
        if result["accuracy"] > 0.9:
            print(f"✅ Training target reached at epoch {epoch}")
            break

# Inference simulation
for i in range(15):
    ml_training_step("predict", "sentiment_classifier", {"batch_size": 32})

# Evaluation
ml_training_step("evaluate", "sentiment_classifier", {})

# Run financial transaction simulation
print("\nRunning financial transaction simulation...")

transaction_scenarios = [
    ("deposit", "user001", 1000),
    ("withdrawal", "user002", 500),
    ("transfer", "user001", 250),
    ("deposit", "user003", 15000),  # Large amount - suspicious
    ("withdrawal", "user002", 75000),  # Over limit - should fail
    *[("transfer", "user004", 100) for _ in range(6)],  # High frequency - suspicious
    ("deposit", "user005", 2000),
    ("withdrawal", "user001", 800),
]

for txn_type, user, amount in transaction_scenarios:
    result = financial_transaction(txn_type, user, amount)

# Generate reports
ml_summary = ml_observer.get_ml_summary()
financial_summary = financial_observer.get_financial_summary()

print(f"\n=== ML Workflow Summary ===")
print(f"Training epochs completed: {ml_summary['training_epochs']}")
print(f"Models evaluated: {ml_summary['models_evaluated']}")
print(f"Average inference times: {ml_summary['avg_inference_times']}")
print(f"Latest metrics: {ml_summary['latest_metrics']}")

print(f"\n=== Financial Audit Summary ===")
print(f"Total transactions: {financial_summary['total_transactions']}")
print(f"Success rate: {financial_summary['success_rate']:.1f}%")
print(f"Total amount processed: ${financial_summary['total_amount']:,.2f}")
print(f"Suspicious activities detected: {financial_summary['suspicious_activity']}")


=== Custom Domain-Specific Observers ===
Running ML workflow simulation...
📊 Training: Epoch 0, Accuracy: 0.699, Loss: 1.985
📊 Training: Epoch 1, Accuracy: 0.707, Loss: 1.902
📊 Training: Epoch 2, Accuracy: 0.674, Loss: 1.876
📊 Training: Epoch 3, Accuracy: 0.712, Loss: 1.598
📊 Training: Epoch 4, Accuracy: 0.883, Loss: 1.238
📊 Training: Epoch 5, Accuracy: 0.864, Loss: 1.334
📊 Training: Epoch 6, Accuracy: 0.950, Loss: 0.946
✅ Training target reached at epoch 6
🔍 SLOW INFERENCE: sentiment_classifier took 182ms
🔍 SLOW INFERENCE: sentiment_classifier took 168ms
🔍 SLOW INFERENCE: sentiment_classifier took 179ms
🔍 SLOW INFERENCE: sentiment_classifier took 136ms
🔍 SLOW INFERENCE: sentiment_classifier took 161ms
🔍 SLOW INFERENCE: sentiment_classifier took 172ms


[2025-06-14 18:40:10,303] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: financial_transaction] [Module: __main__] [Error Type: PermissionError] [Error Message: Withdrawal limit exceeded] [Arguments: {"transaction_type": "withdrawal", "user_id": "user002", "amount": 75000, "kwargs": {}}] [Timestamp: 1749919210.30] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'transaction_failed', 'amount': 0}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'transaction_failed', 'amount': 0}] 


[2025-06-14 18:40:10,304] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for financial_transaction | PermissionError: Withdrawal limit exceeded 

Traceback (most recent call last):
  File "/home/adhd/src/personal/observer-pattern/callpyback/core/decorator.py", line 188, in 

📈 Evaluation: {'accuracy': 0.912485383725438, 'precision': 0.8686626867214345, 'recall': 0.8506144513927338, 'f1_score': 0.7877708384317763}

Running financial transaction simulation...
🚨 SUSPICIOUS TRANSACTION: TXN_000004 - $15000 by user003
🚨 SUSPICIOUS TRANSACTION: TXN_000005 - $75000 by user002
🚨 SUSPICIOUS TRANSACTION: TXN_000010 - $100 by user004
🚨 SUSPICIOUS TRANSACTION: TXN_000011 - $100 by user004

=== ML Workflow Summary ===
Training epochs completed: 7
Models evaluated: 4
Average inference times: {'sentiment_classifier': 0.09250733057657877}
Latest metrics: {'accuracy': 0.912485383725438, 'precision': 0.8686626867214345, 'recall': 0.8506144513927338, 'f1_score': 0.7877708384317763}

=== Financial Audit Summary ===
Total transactions: 13
Success rate: 92.3%
Total amount processed: $20,150.00
Suspicious activities detected: 4


## 8. Advanced Integration Patterns {#integration}

### Microservices and Distributed System Patterns

In [12]:
def distributed_systems_demo():
    """Demonstrate patterns for distributed systems and microservices."""
    print("\n=== Distributed Systems Integration ===")

    # Service mesh observer for microservices
    class ServiceMeshObserver(BaseObserver):
        def __init__(self):
            super().__init__(priority=85, name="ServiceMesh")
            self.service_calls = []
            self.service_health = defaultdict(lambda: {"healthy": 0, "failed": 0})
            self.latency_percentiles = defaultdict(list)

        def update(self, context):
            if context.state == ExecutionState.COMPLETED:
                service_name = context.arguments.get("service", "unknown")

                call_record = {
                    "service": service_name,
                    "timestamp": context.timestamp,
                    "success": context.is_successful,
                    "latency": (
                        getattr(context.result, "execution_time", 0)
                        if context.result
                        else 0
                    ),
                    "caller": context.arguments.get("caller_service", "unknown"),
                }

                self.service_calls.append(call_record)

                # Update service health metrics
                if context.is_successful:
                    self.service_health[service_name]["healthy"] += 1
                else:
                    self.service_health[service_name]["failed"] += 1

                # Track latency percentiles
                if call_record["latency"] > 0:
                    self.latency_percentiles[service_name].append(
                        call_record["latency"]
                    )

                # Alert on service issues
                total_calls = (
                    self.service_health[service_name]["healthy"]
                    + self.service_health[service_name]["failed"]
                )

                if total_calls >= 10:
                    error_rate = (
                        self.service_health[service_name]["failed"] / total_calls
                    ) * 100
                    if error_rate > 20:  # 20% error rate threshold
                        print(
                            f"🚨 SERVICE ALERT: {service_name} error rate: {error_rate:.1f}%"
                        )

        def get_service_mesh_metrics(self):
            service_metrics = {}

            for service, health in self.service_health.items():
                total = health["healthy"] + health["failed"]
                latencies = self.latency_percentiles.get(service, [])

                service_metrics[service] = {
                    "total_calls": total,
                    "error_rate": (health["failed"] / max(total, 1)) * 100,
                    "avg_latency": sum(latencies) / max(len(latencies), 1),
                    "p95_latency": (
                        sorted(latencies)[int(0.95 * len(latencies))]
                        if latencies
                        else 0
                    ),
                    "health_status": (
                        "healthy"
                        if (health["failed"] / max(total, 1)) < 0.1
                        else "degraded"
                    ),
                }

            return service_metrics

    # Distributed tracing observer
    class DistributedTracingObserver(BaseObserver):
        def __init__(self):
            super().__init__(priority=80, name="DistributedTracing")
            self.traces = {}
            self.active_spans = {}

        def update(self, context):
            trace_id = context.arguments.get("trace_id", f"trace_{time.time()}")
            span_id = context.arguments.get(
                "span_id", f"span_{random.randint(1000, 9999)}"
            )

            if context.state == ExecutionState.PRE_EXECUTION:
                # Start span
                span = {
                    "trace_id": trace_id,
                    "span_id": span_id,
                    "service": context.arguments.get("service", "unknown"),
                    "operation": context.function_signature.name,
                    "start_time": context.timestamp,
                    "parent_span": context.arguments.get("parent_span"),
                }
                self.active_spans[span_id] = span

                if trace_id not in self.traces:
                    self.traces[trace_id] = []

            elif context.state == ExecutionState.COMPLETED:
                # Complete span
                if span_id in self.active_spans:
                    span = self.active_spans[span_id]
                    span.update(
                        {
                            "end_time": context.timestamp,
                            "duration": (
                                getattr(context.result, "execution_time", 0)
                                if context.result
                                else 0
                            ),
                            "success": context.is_successful,
                            "error": (
                                str(context.result.exception)
                                if context.is_failed
                                else None
                            ),
                        }
                    )

                    self.traces[trace_id].append(span)
                    del self.active_spans[span_id]

        def get_trace_summary(self):
            return {
                "total_traces": len(self.traces),
                "active_spans": len(self.active_spans),
                "completed_spans": sum(len(spans) for spans in self.traces.values()),
            }

    # Setup distributed system observers
    service_mesh = ServiceMeshObserver()
    distributed_tracing = DistributedTracingObserver()

    # Microservice simulation functions
    @CallPyBack(
        observers=[service_mesh, distributed_tracing],
        exception_classes=(ConnectionError, TimeoutError, ValueError),
        default_return={"status": "service_unavailable"},
    )
    def microservice_call(service, operation, data, **kwargs):
        """Simulate microservice call with distributed tracing."""

        # Simulate service-specific behavior
        if service == "user_service":
            if operation == "get_user":
                time.sleep(random.uniform(0.01, 0.05))
                if random.random() < 0.05:  # 5% failure rate
                    raise ConnectionError("User service database connection failed")
                return {
                    "user_id": data.get("user_id"),
                    "name": f"User_{data.get('user_id')}",
                }

        elif service == "order_service":
            if operation == "create_order":
                time.sleep(random.uniform(0.02, 0.08))
                if random.random() < 0.1:  # 10% failure rate
                    raise ValueError("Invalid order data")
                return {
                    "order_id": f"order_{random.randint(1000, 9999)}",
                    "status": "created",
                }

        elif service == "payment_service":
            if operation == "process_payment":
                time.sleep(random.uniform(0.05, 0.15))  # Slower payment processing
                if random.random() < 0.15:  # 15% failure rate
                    raise TimeoutError("Payment gateway timeout")
                return {
                    "payment_id": f"pay_{random.randint(1000, 9999)}",
                    "status": "processed",
                }

        elif service == "notification_service":
            if operation == "send_notification":
                time.sleep(random.uniform(0.01, 0.03))
                if random.random() < 0.02:  # 2% failure rate
                    raise ConnectionError("SMTP server unavailable")
                return {
                    "notification_id": f"notif_{random.randint(1000, 9999)}",
                    "sent": True,
                }

        return {"status": "completed", "service": service}

    # Distributed workflow orchestration
    @CallPyBack(
        observers=[distributed_tracing],
        exception_classes=(Exception,),
        default_return={"status": "workflow_failed"},
    )
    def distributed_workflow(workflow_type, user_id, **kwargs):
        """Orchestrate distributed workflow across microservices."""
        trace_id = f"workflow_{user_id}_{int(time.time())}"

        try:
            if workflow_type == "user_registration":
                # Step 1: Create user
                user_result = microservice_call(
                    "user_service",
                    "create_user",
                    {"user_id": user_id, "email": kwargs.get("email")},
                    trace_id=trace_id,
                    span_id=f"span_user_{user_id}",
                )

                # Step 2: Send welcome notification
                notification_result = microservice_call(
                    "notification_service",
                    "send_notification",
                    {"user_id": user_id, "type": "welcome"},
                    trace_id=trace_id,
                    span_id=f"span_notif_{user_id}",
                    parent_span=f"span_user_{user_id}",
                )

                return {
                    "status": "user_registered",
                    "user": user_result,
                    "notification": notification_result,
                }

            elif workflow_type == "order_processing":
                # Step 1: Get user info
                user_result = microservice_call(
                    "user_service",
                    "get_user",
                    {"user_id": user_id},
                    trace_id=trace_id,
                    span_id=f"span_get_user_{user_id}",
                )

                # Step 2: Create order
                order_result = microservice_call(
                    "order_service",
                    "create_order",
                    {"user_id": user_id, "items": kwargs.get("items", [])},
                    trace_id=trace_id,
                    span_id=f"span_order_{user_id}",
                    parent_span=f"span_get_user_{user_id}",
                )

                # Step 3: Process payment
                payment_result = microservice_call(
                    "payment_service",
                    "process_payment",
                    {
                        "order_id": order_result.get("order_id"),
                        "amount": kwargs.get("amount", 100),
                    },
                    trace_id=trace_id,
                    span_id=f"span_payment_{user_id}",
                    parent_span=f"span_order_{user_id}",
                )

                # Step 4: Send confirmation
                notification_result = microservice_call(
                    "notification_service",
                    "send_notification",
                    {
                        "user_id": user_id,
                        "type": "order_confirmation",
                        "order_id": order_result.get("order_id"),
                    },
                    trace_id=trace_id,
                    span_id=f"span_confirm_{user_id}",
                    parent_span=f"span_payment_{user_id}",
                )

                return {
                    "status": "order_completed",
                    "user": user_result,
                    "order": order_result,
                    "payment": payment_result,
                    "notification": notification_result,
                }

        except Exception as e:
            print(f"Workflow failed: {e}")
            raise

    # Run distributed system simulation
    print("Simulating distributed microservices architecture...")

    # Test individual services
    print("Testing individual microservices...")
    service_tests = [
        ("user_service", "get_user", {"user_id": "user123"}),
        (
            "order_service",
            "create_order",
            {"user_id": "user123", "items": ["item1", "item2"]},
        ),
        (
            "payment_service",
            "process_payment",
            {"order_id": "order123", "amount": 99.99},
        ),
        (
            "notification_service",
            "send_notification",
            {"user_id": "user123", "type": "welcome"},
        ),
    ]

    for service, operation, data in (
        service_tests * 3
    ):  # Run multiple times for statistics
        result = microservice_call(
            service,
            operation,
            data,
            caller_service="api_gateway",
            trace_id=f"test_{service}_{time.time()}",
        )

    # Test distributed workflows
    print("Testing distributed workflows...")
    workflow_tests = [
        ("user_registration", "user001", {"email": "user001@example.com"}),
        (
            "order_processing",
            "user002",
            {"items": ["item1", "item2"], "amount": 149.99},
        ),
        ("user_registration", "user003", {"email": "user003@example.com"}),
        ("order_processing", "user001", {"items": ["item3"], "amount": 49.99}),
        (
            "order_processing",
            "user004",
            {"items": ["item1", "item4", "item5"], "amount": 299.99},
        ),
    ]

    workflow_results = []
    for workflow_type, user_id, kwargs in workflow_tests:
        result = distributed_workflow(workflow_type, user_id, **kwargs)
        workflow_results.append(result)

    # Generate distributed systems report
    service_metrics = service_mesh.get_service_mesh_metrics()
    trace_summary = distributed_tracing.get_trace_summary()

    print(f"\n=== Distributed Systems Report ===")
    print(f"Service Mesh Metrics:")
    for service, metrics in service_metrics.items():
        print(f"  {service}:")
        print(f"    Total calls: {metrics['total_calls']}")
        print(f"    Error rate: {metrics['error_rate']:.1f}%")
        print(f"    Avg latency: {metrics['avg_latency']*1000:.1f}ms")
        print(f"    P95 latency: {metrics['p95_latency']*1000:.1f}ms")
        print(f"    Health: {metrics['health_status']}")

    print(f"\nDistributed Tracing Summary:")
    print(f"  Total traces: {trace_summary['total_traces']}")
    print(f"  Completed spans: {trace_summary['completed_spans']}")
    print(f"  Active spans: {trace_summary['active_spans']}")

    successful_workflows = sum(
        1
        for r in workflow_results
        if isinstance(r, dict)
        and r.get("status") in ["user_registered", "order_completed"]
    )
    print(
        f"\nWorkflow Success Rate: {(successful_workflows/len(workflow_results))*100:.1f}%"
    )

    return service_mesh, distributed_tracing, workflow_results


service_mesh, tracing, workflows = distributed_systems_demo()


=== Distributed Systems Integration ===
Simulating distributed microservices architecture...
Testing individual microservices...


[2025-06-14 18:40:11,292] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: microservice_call] [Module: __main__] [Error Type: ValueError] [Error Message: Invalid order data] [Arguments: {"service": "order_service", "operation": "create_order", "data": {"user_id": "user002", "items": []}, "kwargs": {"trace_id": "workflow_user002_1749919211", "span_id": "span_order_user002", "parent...] [Timestamp: 1749919211.23] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'service_unavailable'}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'service_unavailable'}] 


[2025-06-14 18:40:11,293] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for microservice_call | ValueError: Invalid order data 

Traceback (most recent call last):
  File "/home/adhd/src/personal/obse

Testing distributed workflows...


[2025-06-14 18:40:11,447] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:557} ERROR - DEFAULT ERROR HANDLER | Unhandled error caught by fallback handler [Function: microservice_call] [Module: __main__] [Error Type: ConnectionError] [Error Message: SMTP server unavailable] [Arguments: {"service": "notification_service", "operation": "send_notification", "data": {"user_id": "user003", "type": "welcome"}, "kwargs": {"trace_id": "workflow_user003_1749919211", "span_id": "span_notif...] [Timestamp: 1749919211.42] [Handler Type: Fallback/Catch-all] [Default Return: {'status': 'service_unavailable'}] [Stack Trace Available: True] [Action: Returning configured default value: {'status': 'service_unavailable'}] 


[2025-06-14 18:40:11,448] {/home/adhd/src/personal/observer-pattern/callpyback/management/error_handling.py:572} ERROR - STACK TRACE for microservice_call | ConnectionError: SMTP server unavailable 

Traceback (most recent call last):
  File "/home/ad


=== Distributed Systems Report ===
Service Mesh Metrics:
  user_service:
    Total calls: 8
    Error rate: 12.5%
    Avg latency: 23.7ms
    P95 latency: 45.8ms
    Health: degraded
  order_service:
    Total calls: 6
    Error rate: 16.7%
    Avg latency: 52.9ms
    P95 latency: 65.0ms
    Health: degraded
  payment_service:
    Total calls: 6
    Error rate: 16.7%
    Avg latency: 96.8ms
    P95 latency: 145.4ms
    Health: degraded
  notification_service:
    Total calls: 8
    Error rate: 12.5%
    Avg latency: 23.3ms
    P95 latency: 30.0ms
    Health: degraded

Distributed Tracing Summary:
  Total traces: 0
  Completed spans: 0
  Active spans: 0

Workflow Success Rate: 100.0%


## Summary

This notebook demonstrates the advanced capabilities of the CallPyBack framework:

### Key Features Demonstrated:

1. **Observer Patterns**: Variable extraction, state monitoring, and callback chaining
2. **Error Handling**: Circuit breakers, graceful degradation, and error recovery
3. **Performance Monitoring**: Execution time tracking, anomaly detection, and alerting
4. **Multi-Threading**: Thread-safe execution, race condition detection, and concurrent processing
5. **Production Patterns**: Enterprise auditing, security monitoring, and compliance tracking
6. **Custom Observers**: Domain-specific monitoring for ML workflows and financial transactions
7. **Distributed Systems**: Service mesh monitoring, distributed tracing, and workflow orchestration

### Advanced Use Cases:

- **Production Monitoring**: Real-time performance tracking and alerting
- **Security Auditing**: Comprehensive logging and suspicious activity detection
- **Distributed Tracing**: End-to-end request tracking across microservices
- **Circuit Breaker**: Fault tolerance and service protection
- **Race Condition Detection**: Thread safety validation and concurrency testing

The CallPyBack framework provides a robust foundation for building observable, resilient, and maintainable Python applications with comprehensive monitoring and error handling capabilities.